In [1]:
import mlflow

# Set your tracking URI
mlflow.set_tracking_uri('https://dagshub.com/Pandharimaske/MLOPs_Project.mlflow')

# List recent runs to verify your run_id exists
experiment = mlflow.get_experiment_by_name("my-dvc-pipeline")  # or your experiment name
runs = mlflow.search_runs(experiment_ids=[experiment.experiment_id])
print(runs[['run_id', 'status']])

                             run_id    status
0  48ec05b08322474592586650fefc4362  FINISHED
1  24a3fb096e4e4d65b66e9572d3df0433  FINISHED
2  c9cfd12637064c899c83ff68f7b7f86f  FINISHED
3  996ba3267de64c9cab64eef910235dc1  FINISHED
4  d15df470cdd64adcb9e0423754fff10d  FINISHED


In [2]:
import mlflow

# Replace with your actual run_id from experiment_info.json
run_id = "48ec05b08322474592586650fefc4362"
client = mlflow.tracking.MlflowClient()

# List artifacts for the run
artifacts = client.list_artifacts(run_id)
print("Artifacts in run:")
for artifact in artifacts:
    print(f"  - {artifact.path}")

Artifacts in run:
  - metrics.json
  - models


In [ ]:
def register_model(model_name: str, model_info: dict):
    """Register the model to the MLflow Model Registry."""
    try:
        # More robust model URI construction
        run_id = model_info['run_id']
        model_path = model_info.get('model_path', 'model')  # default to 'model'
        
        # Verify the run exists first
        client = mlflow.tracking.MlflowClient()
        try:
            run = client.get_run(run_id)
            logging.info(f"Found run: {run_id}")
        except Exception as e:
            logging.error(f"Run {run_id} not found: {e}")
            raise
        
        # List artifacts to verify model exists
        artifacts = client.list_artifacts(run_id, model_path)
        if not artifacts:
            logging.error(f"No model found at path '{model_path}' in run {run_id}")
            # Try to list all artifacts
            all_artifacts = client.list_artifacts(run_id)
            logging.info("Available artifacts:")
            for artifact in all_artifacts:
                logging.info(f"  - {artifact.path}")
            raise FileNotFoundError(f"Model not found at {model_path}")
        
        model_uri = f"runs:/{run_id}/{model_path}"
        logging.info(f"Attempting to register model from URI: {model_uri}")
        
        # Register the model
        model_version = mlflow.register_model(model_uri, model_name)
        
        # Transition the model to "Staging" stage
        client.transition_model_version_stage(
            name=model_name,
            version=model_version.version,
            stage="Staging"
        )
        
        logging.info(f'Model {model_name} version {model_version.version} registered and transitioned to Staging.')
        
    except Exception as e:
        logging.error('Error during model registration: %s', e)
        raise